## 7. N-HiTS Baseline

In [4]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [5]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import NHiTSPredictor, HORIZONS

MODEL_NAME = 'NHiTS'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'nhits'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nhits_stability_summary.json'

In [6]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [8]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = NHiTSPredictor(context_length=sample_cgm.shape[1])

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'nhits_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'nhits_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — NHiTS | seed=7

device: cpu  model: NHiTSPredictor  params: 70,002


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.00 | MAE 7.37 | gRMSE 12.05 | Zone A 97.27% | A+B 99.86%
 30 min | RMSE 18.22 | MAE 12.01 | gRMSE 20.95 | Zone A 90.70% | A+B 99.61%
 60 min | RMSE 26.83 | MAE 17.77 | gRMSE 32.46 | Zone A 81.99% | A+B 99.14%
 90 min | RMSE 32.01 | MAE 21.55 | gRMSE 39.52 | Zone A 75.75% | A+B 98.66%
120 min | RMSE 34.98 | MAE 23.78 | gRMSE 43.64 | Zone A 71.80% | A+B 98.32%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=14

device: cpu  model: NHiTSPredictor  params: 70,002


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.24 | MAE 7.52 | gRMSE 12.32 | Zone A 97.15% | A+B 99.85%
 30 min | RMSE 18.51 | MAE 12.13 | gRMSE 21.35 | Zone A 90.76% | A+B 99.53%
 60 min | RMSE 27.28 | MAE 17.81 | gRMSE 33.30 | Zone A 82.16% | A+B 98.96%
 90 min | RMSE 32.49 | MAE 21.41 | gRMSE 40.50 | Zone A 76.37% | A+B 98.44%
120 min | RMSE 35.29 | MAE 23.61 | gRMSE 44.24 | Zone A 72.21% | A+B 98.18%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=21

device: cpu  model: NHiTSPredictor  params: 70,002


...

 15 min | RMSE 11.10 | MAE 7.48 | gRMSE 12.27 | Zone A 97.10% | A+B 99.83%
 30 min | RMSE 18.39 | MAE 12.15 | gRMSE 21.38 | Zone A 90.62% | A+B 99.57%
 60 min | RMSE 27.18 | MAE 17.74 | gRMSE 33.35 | Zone A 82.32% | A+B 99.06%
 90 min | RMSE 32.50 | MAE 21.35 | gRMSE 40.72 | Zone A 76.45% | A+B 98.48%
120 min | RMSE 35.43 | MAE 23.50 | gRMSE 44.84 | Zone A 73.00% | A+B 98.03%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=63

device: cpu  model: NHiTSPredictor  params: 70,002


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.28 | MAE 7.52 | gRMSE 12.51 | Zone A 97.22% | A+B 99.86%
 30 min | RMSE 18.53 | MAE 12.18 | gRMSE 21.53 | Zone A 90.81% | A+B 99.58%
 60 min | RMSE 27.13 | MAE 17.93 | gRMSE 33.03 | Zone A 82.38% | A+B 99.11%
 90 min | RMSE 32.40 | MAE 21.72 | gRMSE 40.29 | Zone A 75.67% | A+B 98.55%
120 min | RMSE 35.48 | MAE 23.92 | gRMSE 44.70 | Zone A 71.68% | A+B 98.18%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=70

device: cpu  model: NHiTSPredictor  params: 70,002


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.16 | MAE 7.51 | gRMSE 12.18 | Zone A 97.14% | A+B 99.84%
 30 min | RMSE 18.37 | MAE 12.16 | gRMSE 21.16 | Zone A 90.66% | A+B 99.59%
 60 min | RMSE 26.96 | MAE 17.94 | gRMSE 32.78 | Zone A 81.83% | A+B 99.06%
 90 min | RMSE 32.12 | MAE 21.56 | gRMSE 39.96 | Zone A 75.82% | A+B 98.62%
120 min | RMSE 35.02 | MAE 23.77 | gRMSE 43.85 | Zone A 71.77% | A+B 98.34%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )


=== Validation (best checkpoint) ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    10.80 |     7.23 |    11.72 |      99.71
           30 |    17.38 |    11.51 |    19.57 |      99.34
           60 |    24.96 |    16.59 |    29.51 |      98.64
           90 |    29.31 |    19.76 |    35.33 |      98.16
          120 |    31.89 |    21.88 |    38.79 |      97.97

=== Test ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.05 |     7.42 |    12.17 |      99.87
           30 |    18.31 |    12.06 |    21.13 |      99.60
           60 |    27.04 |    17.83 |    32.84 |      99.14
           90 |    32.30 |    21.62 |    40.06 |      98.61
          120 |    35.31 |    23.87 |    44.24 |      98.25


In [ ]:
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/nhits_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')